# UpstreamDrift Architecture Overview

This notebook validates the current `shared_python` architecture and demonstrates the core modules used across the golf simulation suite.

## Modules demonstrated
- `src.shared.python.physics.aerodynamics` — aerodynamic force model
- `src.shared.python.physics.topography` — terrain elevation models
- `src.shared.python.logging_pkg.logging_config` — centralised logging
- `src.shared.python.core.constants` — physical constants
- `src.shared.python.physics.physics_parameters` — parameter registry

Run all cells top-to-bottom. No engine (MuJoCo/Pinocchio/Drake) installation required.

In [ ]:
# Cell 1 — Imports and logging setup
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

# Ensure repo root is on sys.path so src.shared.python is importable
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.shared.python.logging_pkg.logging_config import get_logger, setup_logging

setup_logging()
logger = get_logger(__name__)
logger.info("Logging initialised")
print("Cell 1 OK: imports and logging")

In [ ]:
# Cell 2 — Physical constants
from src.shared.python.core.constants import GRAVITY_M_S2

print(f"GRAVITY_M_S2 = {GRAVITY_M_S2} m/s²")
assert GRAVITY_M_S2 > 0, "Gravity must be positive"
logger.info(f"GRAVITY_M_S2={GRAVITY_M_S2}")
print("Cell 2 OK: constants")

In [ ]:
# Cell 3 — Aerodynamics engine
from src.shared.python.physics.aerodynamics import (
    AerodynamicsConfig,
    AerodynamicsEngine,
)

config = AerodynamicsConfig()
engine = AerodynamicsEngine(config=config)

# Compute forces for a 60 m/s drive with 300 rad/s backspin
velocity = np.array([60.0, 0.0, 0.0])
spin = np.array([0.0, 300.0, 0.0])
forces = engine.compute_forces(velocity, spin)

drag_n = float(np.linalg.norm(forces["drag"]))
lift_n = float(np.linalg.norm(forces["lift"]))
total_n = float(np.linalg.norm(forces["total"]))

print(f"Drag : {drag_n:.4f} N")
print(f"Lift : {lift_n:.4f} N")
print(f"Total: {total_n:.4f} N")
assert drag_n > 0, "Drag must be positive"
logger.info(f"Aerodynamics OK — drag={drag_n:.3f}N lift={lift_n:.3f}N")
print("Cell 3 OK: aerodynamics")

In [ ]:
# Cell 4 — Topography models
from src.shared.python.physics.topography import (
    create_flat_terrain,
    create_sloped_terrain,
    create_undulating_terrain,
)

flat = create_flat_terrain(width=100.0, height=100.0, elevation=5.0)
sloped = create_sloped_terrain(
    width=100.0,
    height=100.0,
    slope_direction=np.array([1.0, 0.0]),
    slope_magnitude=0.05,
    base_elevation=0.0,
)
undulating = create_undulating_terrain(
    width=100.0,
    height=100.0,
    amplitude=2.0,
    wavelength=30.0,
    base_elevation=0.0,
)

point = np.array([50.0, 50.0])
flat_elev = flat.get_elevation_at(point)
sloped_elev = sloped.get_elevation_at(point)
undulating_elev = undulating.get_elevation_at(point)

print(f"Flat elevation at (50,50)     : {flat_elev:.3f} m")
print(f"Sloped elevation at (50,50)   : {sloped_elev:.3f} m")
print(f"Undulating elevation at (50,50): {undulating_elev:.3f} m")
assert abs(flat_elev - 5.0) < 0.01, f"Flat terrain must return ~5.0 m, got {flat_elev}"
logger.info("Topography models OK")
print("Cell 4 OK: topography")

In [ ]:
# Cell 5 — Physics parameter registry
from src.shared.python.physics.physics_parameters import ParameterCategory, get_registry

registry = get_registry()
gravity_param = registry.get("GRAVITY")
assert gravity_param is not None, "GRAVITY parameter must exist in registry"
print(f"Registry GRAVITY: {gravity_param.value} {gravity_param.unit}")

ball_params = registry.get_by_category(ParameterCategory.BALL)
print(f"Ball parameter count: {len(ball_params)}")
for p in ball_params:
    print(f"  {p.name}: {p.value} {p.unit}")

logger.info(f"Registry OK — {len(ball_params)} ball parameters")
print("Cell 5 OK: parameter registry")

## Architecture Summary

All five core modules validated:

| Module | Status |
|--------|--------|
| `logging_pkg.logging_config` | OK |
| `core.constants` | OK |
| `physics.aerodynamics` | OK |
| `physics.topography` | OK |
| `physics.physics_parameters` | OK |

See `01_basic_simulation.py`, `02_parameter_sweeps.py`, and `03_injury_risk_tutorial.py` for full worked examples.